# 2026 COMP90042 Project
*Make sure you change the file name with your group id.*

# Readme

## Pipeline Overview

This notebook implements a **two-stage automated fact-checking pipeline**:

1. **Stage 1 — Evidence Retrieval:** Hybrid BM25 + CrossEncoder retrieval indexes `evidence.json` and retrieves the top-k most relevant passages for each claim. BM25 first retrieves a broad candidate set (top_k×3); a CrossEncoder re-ranks by joint cross-attention relevance (entailment-aware, not topic-similarity).
2. **Stage 2 — Claim Verification:** A DeBERTa-v3-base NLI model (`MoritzLaurer/DeBERTa-v3-base-mnli-fever-anli`) classifies the claim against retrieved evidence. Section 2.4 fine-tunes this model on the training set with **class-weighted loss** and **NEI pseudo-evidence sampling** to address label imbalance and the NEI collapse problem.

## Upgrades from Baseline

| Component | Baseline | Upgraded |
|-----------|----------|----------|
| Retrieval | BM25 only, top_k=5 | Hybrid BM25 + CrossEncoder re-rank, top_k=10 |
| BM25 preprocessing | lowercase + punctuation strip | + stopword removal + Porter stemming |
| Re-ranker | None | CrossEncoder (cross-attention, entailment-aware) |
| Classification | Zero-shot NLI (uniform loss) | Fine-tuned 4-class with class-weighted loss |
| NEI training signal | Gold evidence for all labels (NEI F1=0) | BM25 pseudo-evidence for NEI claims (P1) |
| Evidence scoring | Per-passage + average | Top-3 concatenated (multi-evidence, P3) |
| DISPUTED detection | If-then threshold rule (spec violation) | 4th output class via fine-tuned argmax (P0) |
| Training | 3 epochs, fixed | 5 epochs, early stopping (patience=2) on dev macro-F1 |

## Notes for Marker

- OOP class definitions are in **Section 0** so **Run All** works top-to-bottom without undefined errors.
- The model is loaded in **fp16** for inference — designed for Google Colab free tier (NVIDIA T4, 15 GB VRAM).
- Primary evaluation metric: **Harmonic Mean** of claim classification accuracy and evidence retrieval F-score (as per `eval.py`).
- `DISPUTED` is a genuine **4th output class** predicted by the fine-tuned 4-way softmax head — no hand-crafted threshold rule, complying with the spec's prohibition on if-then classification rules.
- Multi-evidence concatenation (`top-3 passages [SEP]-joined`) lets DeBERTa reason across conflicting signals simultaneously, which is important for DISPUTED and REFUTES.
- Early stopping saves the best checkpoint by dev macro-F1 and reloads it after training.

## 0. Setup & Class Definitions

Run this section **first**. Mounts Google Drive, installs dependencies, imports libraries, and defines all pipeline classes.

### 0.1 Mount Google Drive & Set Working Directory

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

# Update this path to match where you uploaded the folder in your Google Drive
BASE_DIR = "/content/drive/MyDrive/Final_Assignment"
os.chdir(BASE_DIR)
print(f"Working directory: {os.getcwd()}")
print("Files:", os.listdir("."))

### 0.2 Install Dependencies

In [ ]:
!pip install rank_bm25 sentence-transformers nltk peft -q

### 0.3 Imports

In [ ]:
import json
import re
import os
import numpy as np
import torch
from collections import Counter
from rank_bm25 import BM25Okapi
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup,
)
from sklearn.metrics import classification_report, f1_score

import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from sentence_transformers import CrossEncoder  # P2: CrossEncoder re-ranker

nltk.download('stopwords', quiet=True)
STOP_WORDS = set(stopwords.words('english'))
STEMMER    = PorterStemmer()

### 0.4 Class: BM25Retriever

**Upgrades (P1):**
- `_preprocess` now applies stopword removal and Porter stemming in addition to lowercasing and punctuation stripping — improves BM25 term matching quality.
- `top_k` default raised from 5 → 10 for higher recall.

In [ ]:
class BM25Retriever:
    """BM25-based evidence retrieval over the flat evidence corpus.

    PATCH P1: _preprocess now removes stopwords and applies Porter stemming
    for better term-matching quality. Default top_k raised to 10.
    """

    def __init__(self, evidence_dict):
        self.ids   = list(evidence_dict.keys())
        self.texts = list(evidence_dict.values())
        self.bm25  = None

    def _preprocess(self, text):
        # PATCH P1: stopword removal + stemming on top of original lowercase + punct strip
        text   = text.lower()
        text   = re.sub(r"[^\w\s]", " ", text)
        tokens = text.split()
        tokens = [STEMMER.stem(t) for t in tokens if t not in STOP_WORDS]
        return tokens

    def build_index(self):
        tokenized = [self._preprocess(t) for t in self.texts]
        self.bm25 = BM25Okapi(tokenized)
        print(f"BM25 index built over {len(self.texts):,} passages")

    def retrieve(self, claim, top_k=10):  # PATCH P1: default top_k 5 -> 10
        if self.bm25 is None:
            raise RuntimeError("Call build_index() first")
        tokens = self._preprocess(claim)
        scores = self.bm25.get_scores(tokens)
        top_idx = scores.argsort()[::-1][:top_k]
        return [
            {"id": self.ids[i], "text": self.texts[i], "score": float(scores[i])}
            for i in top_idx
        ]

### 0.4b Class: HybridRetriever

**P2:** Wraps `BM25Retriever` with a CrossEncoder re-ranker.
- BM25 retrieves a broad candidate set (`bm25_candidates = top_k * 3`).
- `cross-encoder/ms-marco-MiniLM-L6-v2` (22M params) re-ranks by joint cross-attention relevance score — attends to *both* claim and passage simultaneously, providing entailment-aware ranking rather than topic similarity.
- CrossEncoder does **not** pre-encode the corpus; it scores each (claim, passage) pair at query time.
- Final top-k is returned after re-ranking.

**Why CrossEncoder over bi-encoder?** Bi-encoder cosine similarity scores topic overlap, not entailment. CrossEncoder attends jointly to both texts and is trained to score passage relevance for the query — much better aligned with fact-checking needs (Nogueira & Cho, 2019; DeHaven & Scott, 2023).

In [ ]:
class HybridRetriever:
    """BM25 candidate retrieval + CrossEncoder re-ranking.

    P2: Replaces MiniLM bi-encoder (cosine similarity) with CrossEncoder
    (joint cross-attention). CrossEncoder scores (claim, passage) pairs
    jointly, providing entailment-aware ranking rather than topic similarity.
    No corpus pre-encoding needed — scoring is done at query time.
    """

    CE_MODEL = "cross-encoder/ms-marco-MiniLM-L6-v2"

    def __init__(self, evidence_dict, bm25_retriever):
        self.bm25_retriever = bm25_retriever
        self.ids            = list(evidence_dict.keys())
        self.texts          = list(evidence_dict.values())
        self.cross_encoder  = None

    def build_dense_index(self):
        """Load CrossEncoder model. No corpus pre-encoding step."""
        print(f"Loading CrossEncoder: {self.CE_MODEL}")
        self.cross_encoder = CrossEncoder(self.CE_MODEL)
        print("CrossEncoder loaded.")

    def retrieve(self, claim, top_k=10):
        """BM25 retrieves top_k*3 candidates; CrossEncoder re-ranks to top_k."""
        candidates = self.bm25_retriever.retrieve(claim, top_k=top_k * 3)

        if self.cross_encoder is None:
            return candidates[:top_k]

        pairs  = [(claim, c["text"]) for c in candidates]
        scores = self.cross_encoder.predict(pairs, batch_size=32)
        for cand, sc in zip(candidates, scores.tolist()):
            cand["ce_score"] = sc
        ranked = sorted(candidates, key=lambda x: x["ce_score"], reverse=True)
        return ranked[:top_k]

### 0.5 Class: NLIClassifier

In [ ]:
class NLIClassifier:
    """
    Wraps a HuggingFace sequence-classification model for fact-checking.

    label_map translates the model's raw output labels to the four
    fact-check labels.  It is set as an instance variable so it can be
    swapped out after fine-tuning without changing any other code.

    Zero-shot NLI model (3-class):
        label_map = {entailment->SUPPORTS, contradiction->REFUTES, neutral->NOT_ENOUGH_INFO}
    Fine-tuned 4-class model:
        label_map = {supports->SUPPORTS, refutes->REFUTES, ...}  (pass-through)
    """

    DEFAULT_LABEL_MAP = {
        "entailment":    "SUPPORTS",
        "contradiction": "REFUTES",
        "neutral":       "NOT_ENOUGH_INFO",
    }

    def __init__(self, model_name="MoritzLaurer/DeBERTa-v3-base-mnli-fever-anli", device=0):
        self.device = torch.device(
            f"cuda:{device}" if device >= 0 and torch.cuda.is_available() else "cpu"
        )
        print(f"Loading model on {self.device} ...")
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model     = AutoModelForSequenceClassification.from_pretrained(model_name)
        self.model     = self.model.half().to(self.device)
        self.model.eval()
        self.label_map = dict(self.DEFAULT_LABEL_MAP)  # instance copy — safe to mutate
        print("Model ready.")

    def score_single(self, claim, premise):
        """
        Returns {fact_check_label: probability} after applying label_map.
        Works with both 3-class NLI and 4-class fine-tuned models.
        """
        inputs = self.tokenizer(
            premise, claim,
            return_tensors="pt",
            truncation=True,
            max_length=512,
        ).to(self.device)
        with torch.no_grad():
            logits = self.model(**inputs).logits
        probs = torch.softmax(logits.float(), dim=-1).squeeze().cpu().tolist()
        scores = {}
        for i, p in enumerate(probs):
            raw        = self.model.config.id2label[i].lower()
            fact_label = self.label_map.get(raw, "NOT_ENOUGH_INFO")
            scores[fact_label] = scores.get(fact_label, 0.0) + p
        return scores  # e.g. {"SUPPORTS": 0.7, "REFUTES": 0.1, "NOT_ENOUGH_INFO": 0.2}

    def classify_single(self, claim, premise):
        """Returns (fact_check_label, scores_dict) for one pair."""
        scores = self.score_single(claim, premise)
        return max(scores, key=scores.get), scores

### 0.6 Class: FactCheckPipeline

**P0 (spec compliance):** `DISPUTED` is no longer detected via a hand-crafted threshold rule.
The fine-tuned classifier predicts it as a genuine 4th output class — `label = argmax` over the
4-class softmax. No if-then rule is applied.

**P3 (multi-evidence):** Instead of scoring each passage independently and averaging,
the top-3 retrieved passages are concatenated with `[SEP]` and scored as a single input.
This lets DeBERTa reason across conflicting signals simultaneously, which is critical for
DISPUTED (where passages both support and refute) and REFUTES detection.

In [ ]:
class FactCheckPipeline:
    """
    End-to-end pipeline:
      1. Retriever (HybridRetriever) fetches top-k evidence passages.
      2. Top-3 passages are concatenated with [SEP] (multi-evidence, P3).
      3. NLI classifier scores the concatenated evidence against the claim.
      4. Predicted label = argmax over 4-class softmax (P0: no threshold rule).
    Returns (label, evidence_ids) so eval.py receives both outputs.

    predict() supports two evaluation modes for oracle/pipeline gap analysis:
      - Oracle mode (pass gold_evidence_ids): skips retrieval, scores gold passages.
        Measures classifier ceiling. Any error here is a classifier problem.
      - Pipeline mode (default): runs full BM25 → CrossEncoder retrieval.
        Matches what eval.py measures — the real system score.
    Gap = A_oracle − A_pipeline identifies the bottleneck:
      gap > 0.10  → retriever is bottleneck → Priority 2b (supervised CrossEncoder)
      gap < 0.05 and A_oracle < 0.60 → classifier is bottleneck → Priority 4 (LoRA)

    P0: DISPUTED is a genuine 4th output class — argmax over 4-way softmax,
        not a hand-crafted threshold rule (spec compliance).
    P3: Multi-evidence concatenation lets DeBERTa attend jointly to passages
        that may conflict, improving DISPUTED and REFUTES recall.
    """

    def __init__(self, retriever, classifier, top_k=10):
        self.retriever  = retriever
        self.classifier = classifier
        self.top_k      = top_k
        self._ev_lookup = None  # lazy-built for oracle mode

    def _get_ev_lookup(self):
        """Build evidence ID→text lookup from retriever's internal corpus (lazy)."""
        if self._ev_lookup is None:
            self._ev_lookup = {
                eid: self.retriever.texts[i]
                for i, eid in enumerate(self.retriever.ids)
            }
        return self._ev_lookup

    def predict(self, claim, gold_evidence_ids=None):
        """
        Returns (label, evidence_ids_list).

        gold_evidence_ids: list of evidence IDs for oracle mode (dev eval only).
          Pass item["evidences"] to bypass retrieval and score gold passages.
          Omit (or None) for pipeline mode — this is what eval.py measures.
        """
        if gold_evidence_ids is not None:
            # Oracle mode: skip retriever, use provided gold evidence IDs
            ev_lookup = self._get_ev_lookup()
            passages  = [{"id": eid, "text": ev_lookup[eid]}
                         for eid in gold_evidence_ids if eid in ev_lookup]
        else:
            # Pipeline mode: run full BM25 → CrossEncoder retrieval
            passages = self.retriever.retrieve(claim, top_k=self.top_k)

        if not passages:
            return "NOT_ENOUGH_INFO", []

        # P3: concatenate top-3 passages for joint cross-attention reasoning
        combined = " [SEP] ".join(p["text"] for p in passages[:3])
        scores   = self.classifier.score_single(claim, combined)

        ev_ids = [p["id"] for p in passages]
        # P0: argmax over 4-class softmax — no threshold rule
        return max(scores, key=scores.get), ev_ids

    def predict_batch(self, claims, verbose=True):
        """Returns list of (label, evidence_ids) tuples (pipeline mode)."""
        results = []
        for i, claim in enumerate(claims):
            results.append(self.predict(claim))
            if verbose and (i + 1) % 50 == 0:
                print(f"  {i+1}/{len(claims)} processed")
        return results

### 0.7 Class: FactCheckDataset

**P1 (NEI pseudo-evidence sampling):** For `NOT_ENOUGH_INFO` training examples, gold evidence
is replaced with top BM25-retrieved passages. These passages are topically related to the claim
but are not entailment-supporting — which is exactly the signal the model needs to learn NEI.
Without this fix, the model always receives gold evidence even for NEI claims and never learns
when evidence is insufficient, causing NEI F1 = 0.00 (Soleimani et al., 2020).

**P2 (class weights):** `__init__` computes and exposes `class_weights` tensor for use in
weighted cross-entropy loss during fine-tuning.

In [ ]:
class FactCheckDataset(Dataset):
    """
    PyTorch Dataset for fine-tuning.
    Input:  claim [SEP] evidence text
    Output: 4-class label index

    P1: For NOT_ENOUGH_INFO claims, gold evidence is replaced with
        BM25-retrieved passages (topically related, not entailing).
        This teaches the NEI signal instead of hiding it with gold evidence.
    P2: Exposes class_weights for weighted cross-entropy loss.
    """

    LABEL2ID = {"SUPPORTS": 0, "REFUTES": 1, "NOT_ENOUGH_INFO": 2, "DISPUTED": 3}

    def __init__(self, data_dict, evidence_dict, tokenizer,
                 bm25_retriever=None, max_length=512, max_ev=3):
        self.tokenizer  = tokenizer
        self.max_length = max_length
        self.items      = []
        for item in data_dict.values():
            claim    = item["claim_text"]
            label_id = self.LABEL2ID[item["claim_label"]]

            # P1: NEI pseudo-evidence — BM25-retrieved passages instead of gold
            if item["claim_label"] == "NOT_ENOUGH_INFO" and bm25_retriever is not None:
                retrieved = bm25_retriever.retrieve(claim, top_k=max_ev)
                ev_text   = " ".join(r["text"] for r in retrieved)
            else:
                ev_text = " ".join(
                    evidence_dict.get(ev, "") for ev in item.get("evidences", [])[:max_ev]
                )
            self.items.append((claim, ev_text, label_id))

        # Compute class weights (inverse frequency) for weighted loss
        label_counts = Counter(item[2] for item in self.items)
        total        = len(self.items)
        self.class_weights = torch.tensor(
            [total / (label_counts.get(i, 1) * len(self.LABEL2ID)) for i in range(len(self.LABEL2ID))],
            dtype=torch.float
        )
        print(f"Class weights: { {k: f'{self.class_weights[v].item():.3f}' for k,v in self.LABEL2ID.items()} }")

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        claim, evidence, label = self.items[idx]
        enc = self.tokenizer(
            evidence, claim,
            max_length=self.max_length,
            truncation=True,
            padding="max_length",
            return_tensors="pt",
        )
        return (
            {k: v.squeeze(0) for k, v in enc.items()},
            torch.tensor(label, dtype=torch.long),
        )

# 1.DataSet Processing
(You can add as many code blocks and text blocks as you need. However, YOU SHOULD NOT MODIFY the section title)

## 1.1 Load Evidence Corpus

In [ ]:
with open("evidence.json") as f:
    evidence_dict = json.load(f)

print(f"Loaded {len(evidence_dict):,} evidence passages")
for k in list(evidence_dict.keys())[:3]:
    print(f"  {k}: {evidence_dict[k][:90]}")

## 1.2 Load Dataset Splits

In [ ]:
def load_json_safe(path):
    if not os.path.exists(path):
        print(f"Warning: {path} not found")
        return {}
    with open(path) as f:
        return json.load(f)

train_data = load_json_safe("train-claims.json")
dev_data   = load_json_safe("dev-claims.json")
test_data  = load_json_safe("test-claims-unlabelled.json")

print(f"Train: {len(train_data):,} | Dev: {len(dev_data):,} | Test: {len(test_data):,}")

In [ ]:
# Inspect one example and show label distribution
if train_data:
    first_key = next(iter(train_data))
    print(f"Claim ID: {first_key}")
    print(json.dumps(train_data[first_key], indent=2))

for name, split in [("train", train_data), ("dev", dev_data)]:
    if not split:
        continue
    labels = [v["claim_label"] for v in split.values()]
    print(f"\n{name} distribution ({len(split):,} examples):")
    for lbl, cnt in sorted(Counter(labels).items()):
        print(f"  {lbl:<20} {cnt:>5}  ({cnt/len(labels)*100:.1f}%)")

# 2.Model Implementation
(You can add as many code blocks and text blocks as you need. However, YOU SHOULD NOT MODIFY the section title)

## 2.1 Stage 1 — Build BM25 + CrossEncoder Retrieval Index

CPU-only. Builds a `BM25Okapi` index with improved preprocessing (stopword removal + stemming),
then wraps it with a `HybridRetriever` that loads `cross-encoder/ms-marco-MiniLM-L6-v2` (P2).

CrossEncoder loads in seconds (no corpus pre-encoding). It scores each (claim, passage) pair
jointly at query time, providing entailment-aware ranking.

**Note:** If Colab memory is tight, use `retriever` (BM25-only) in Section 2.3 instead of
`hybrid_retriever` — the pipeline will fall back to BM25-only ranking.

In [ ]:
retriever = BM25Retriever(evidence_dict)
retriever.build_index()

In [ ]:
# P2: build hybrid retriever — loads CrossEncoder (no corpus pre-encoding)
hybrid_retriever = HybridRetriever(evidence_dict, retriever)
hybrid_retriever.build_dense_index()

In [ ]:
# Smoke-test retrieval — compare BM25-only vs Hybrid BM25+CrossEncoder
sample_claim = next(iter(dev_data.values()))["claim_text"]
print(f"Claim: {sample_claim}\n")

print("=== BM25-only (top 5) ===")
for i, p in enumerate(retriever.retrieve(sample_claim, top_k=5), 1):
    print(f"[{i}] bm25_score={p['score']:.3f}  id={p['id']}")
    print(f"     {p['text'][:110]}")

print("\n=== Hybrid BM25+CrossEncoder (top 5) ===")
for i, p in enumerate(hybrid_retriever.retrieve(sample_claim, top_k=5), 1):
    ce = p.get("ce_score", float("nan"))
    print(f"[{i}] ce_score={ce:.3f}  id={p['id']}")
    print(f"     {p['text'][:110]}")

## 2.2 Stage 2 — Initialise Zero-Shot NLI Classifier

Loads `MoritzLaurer/DeBERTa-v3-base-mnli-fever-anli` in fp16 on the T4 GPU.
This is the **zero-shot baseline**. Section 2.4 fine-tunes it on the training set.

In [ ]:
# device=0 -> T4 GPU; set device=-1 to fall back to CPU
classifier = NLIClassifier(
    model_name="MoritzLaurer/DeBERTa-v3-base-mnli-fever-anli",
    device=0
)

In [ ]:
# Smoke-test: check aggregated scores for one pair
scores = classifier.score_single(
    claim="John Bennet Lawes was an English entrepreneur.",
    premise="John Bennet Lawes, English entrepreneur and agricultural scientist"
)
print(f"Fact-check scores: {scores}")
print(f"Predicted label  : {max(scores, key=scores.get)}")

## 2.3 Assemble Zero-Shot Pipeline

`FactCheckPipeline` concatenates the top-3 retrieved passages and scores them jointly (P3).
Label is argmax over the 4-class softmax — no threshold rule (P0).

**P2+P3:** Pipeline uses `hybrid_retriever` (CrossEncoder re-ranked) and multi-evidence
concatenation. Swap `hybrid_retriever` → `retriever` if dense index was skipped.

In [ ]:
# P0: no disputed_threshold — DISPUTED predicted by 4-class argmax after fine-tuning
# P2: hybrid_retriever uses CrossEncoder re-ranking
# P3: FactCheckPipeline concatenates top-3 passages (see class definition)
pipeline = FactCheckPipeline(
    hybrid_retriever,
    classifier,
    top_k=10,
)

# End-to-end sanity check
sample_claim = next(iter(dev_data.values()))["claim_text"]
label, ev_ids = pipeline.predict(sample_claim)
print(f"Claim : {sample_claim}")
print(f"Label : {label}")
print(f"Evid. : {ev_ids}")

## 2.4 Fine-Tune on Training Set

Replaces the 3-class NLI head with a **4-class head** (SUPPORTS / REFUTES / NOT_ENOUGH_INFO / DISPUTED)
and fine-tunes on `train-claims.json`.

- Input: `claim [SEP] concatenated gold evidence` (or BM25 pseudo-evidence for NEI — P1)
- **P0:** DISPUTED is a genuine 4th output class — `argmax` over 4-way softmax, no threshold rule
- **P1:** NEI training examples use BM25-retrieved passages instead of gold evidence, teaching
  the model what "not enough evidence" looks like
- **P2:** Class-weighted cross-entropy prevents collapse to SUPPORTS (44% of training labels)
- **P5:** 5 epochs max with early stopping (patience=2) on dev macro-F1; best checkpoint reloaded
- After training the pipeline `classifier` is hot-swapped to use the fine-tuned weights

In [ ]:
LABEL2ID = {"SUPPORTS": 0, "REFUTES": 1, "NOT_ENOUGH_INFO": 2, "DISPUTED": 3}
ID2LABEL  = {v: k for k, v in LABEL2ID.items()}

ft_device    = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
ft_tokenizer = AutoTokenizer.from_pretrained(
    "MoritzLaurer/DeBERTa-v3-base-mnli-fever-anli"
)

# Load DeBERTa encoder with a fresh 4-class head (ignore_mismatched_sizes
# drops the 3-class NLI head and adds a randomly initialised 4-class head)
ft_model = AutoModelForSequenceClassification.from_pretrained(
    "MoritzLaurer/DeBERTa-v3-base-mnli-fever-anli",
    num_labels=4,
    id2label=ID2LABEL,
    label2id=LABEL2ID,
    ignore_mismatched_sizes=True,
).to(ft_device)

print(f"Fine-tune model on: {ft_device}")
print(f"Output labels     : {ft_model.config.id2label}")

In [ ]:
# P1: pass retriever so FactCheckDataset uses BM25 pseudo-evidence for NEI claims
train_dataset = FactCheckDataset(train_data, evidence_dict, ft_tokenizer,
                                 bm25_retriever=retriever)
train_loader  = DataLoader(train_dataset, batch_size=8, shuffle=True)
print(f"Training examples : {len(train_dataset)}")
print(f"Steps per epoch   : {len(train_loader)}")

In [ ]:
def _eval_macro_f1_dev(model, tokenizer, dev_data, evidence_dict, device, batch_size=16):
    """Dev macro-F1 using gold evidence — fast proxy for early stopping decisions."""
    model.eval()
    LABEL_NAMES = ["SUPPORTS", "REFUTES", "NOT_ENOUGH_INFO", "DISPUTED"]
    ID2LABEL    = {0: "SUPPORTS", 1: "REFUTES", 2: "NOT_ENOUGH_INFO", 3: "DISPUTED"}
    y_true, y_pred = [], []
    items = [
        (v["claim_text"],
         " ".join(evidence_dict.get(ev, "") for ev in v.get("evidences", [])[:3]),
         v["claim_label"])
        for v in dev_data.values()
    ]
    for i in range(0, len(items), batch_size):
        batch    = items[i : i + batch_size]
        claims   = [x[0] for x in batch]
        ev_texts = [x[1] for x in batch]
        y_true.extend(x[2] for x in batch)
        enc = tokenizer(ev_texts, claims, max_length=512,
                        truncation=True, padding=True, return_tensors="pt").to(device)
        with torch.no_grad():
            preds = model(**enc).logits.argmax(dim=-1).cpu().tolist()
        y_pred.extend(ID2LABEL[p] for p in preds)
    return f1_score(y_true, y_pred, average="macro", labels=LABEL_NAMES, zero_division=0)


# P5: 5 epochs, early stopping with patience=2 on dev macro-F1
EPOCHS     = 5
LR         = 2e-5
best_f1    = 0.0
patience   = 2
no_improve = 0

ft_model = ft_model.float().to(ft_device)
use_amp  = ft_device.type == "cuda"

optimizer   = AdamW(ft_model.parameters(), lr=LR, weight_decay=0.01)
total_steps = len(train_loader) * EPOCHS
scheduler   = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps  = total_steps // 10,
    num_training_steps= total_steps,
)
scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

weighted_loss_fn = torch.nn.CrossEntropyLoss(
    weight=train_dataset.class_weights.to(device=ft_device, dtype=torch.float32)
)
print(f"Using weighted loss. Weights: {train_dataset.class_weights.tolist()}")

for epoch in range(EPOCHS):
    ft_model.train()
    total_loss = 0.0
    for batch_enc, batch_labels in train_loader:
        batch_enc    = {k: v.to(ft_device) for k, v in batch_enc.items()}
        batch_labels = batch_labels.to(ft_device)
        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast(device_type=ft_device.type, dtype=torch.float16, enabled=use_amp):
            logits = ft_model(**batch_enc).logits
            loss   = weighted_loss_fn(logits.float(), batch_labels)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(ft_model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)
    dev_f1   = _eval_macro_f1_dev(ft_model, ft_tokenizer, dev_data, evidence_dict, ft_device)
    print(f"Epoch {epoch+1}/{EPOCHS}  loss={avg_loss:.4f}  dev macro-F1={dev_f1:.4f}")

    if dev_f1 > best_f1:
        best_f1    = dev_f1
        no_improve = 0
        ft_model.save_pretrained("deberta-best")
        ft_tokenizer.save_pretrained("deberta-best")
        print(f"  → New best ({best_f1:.4f}). Checkpoint saved.")
    else:
        no_improve += 1
        if no_improve >= patience:
            print(f"Early stop at epoch {epoch+1}. Best dev macro-F1: {best_f1:.4f}")
            break

print(f"Training complete. Best dev macro-F1: {best_f1:.4f}")

In [ ]:
# Reload best checkpoint saved during early stopping
SAVE_DIR     = "deberta-best"
ft_model     = AutoModelForSequenceClassification.from_pretrained(SAVE_DIR).to(ft_device)
ft_tokenizer = AutoTokenizer.from_pretrained(SAVE_DIR)
print(f"Best model reloaded from {os.path.abspath(SAVE_DIR)}")

In [ ]:
# Hot-swap the fine-tuned weights into the existing classifier and pipeline
classifier.tokenizer = ft_tokenizer
classifier.model     = ft_model.half()   # fp16 for efficient inference
classifier.model.eval()

# Fine-tuned model outputs fact-check labels directly — pass-through map
# id2label: {0: SUPPORTS, 1: REFUTES, 2: NOT_ENOUGH_INFO, 3: DISPUTED}
classifier.label_map = {
    "supports":        "SUPPORTS",
    "refutes":         "REFUTES",
    "not_enough_info": "NOT_ENOUGH_INFO",
    "disputed":        "DISPUTED",
}

print("Pipeline uses fine-tuned 4-class classifier. DISPUTED predicted via argmax (P0).")

# Quick sanity check
sample_claim = next(iter(dev_data.values()))["claim_text"]
label, ev_ids = pipeline.predict(sample_claim)
print(f"Sample prediction: {label}")

## 2.5 (Conditional) Priority 2b — Supervised CrossEncoder Fine-Tuning

**Run this section only if gap analysis (Section 3.1) shows `A_oracle − A_pipeline > 0.10`.**

The zero-shot CrossEncoder (`cross-encoder/ms-marco-MiniLM-L6-v2`) ranks passages by general
web-search relevance. Fine-tuning it on `(claim, gold_passage)` positive pairs and
`(claim, bm25_non-gold)` hard-negative pairs teaches entailment relevance — the exact signal
fact-checking needs (Nogueira & Cho, 2019; DeHaven & Scott, 2023).

After completing this section, `hybrid_retriever` is updated with the fine-tuned model.
Re-run Section 3 to measure the improvement.

In [ ]:
# P2b — Step 1: BM25 gold recall diagnostic
# If recall < 0.50, raise top_k to 50 before building training pairs.
hits, total = 0, 0
for item in train_data.values():
    if item["claim_label"] == "NOT_ENOUGH_INFO":
        continue
    gold      = set(item.get("evidences", []))
    retrieved = set(r["id"] for r in retriever.retrieve(item["claim_text"], top_k=30))
    hits  += len(gold & retrieved)
    total += len(gold)

recall = hits / total if total > 0 else 0.0
print(f"BM25 top-30 gold recall: {recall:.3f}")
if recall < 0.50:
    print("WARNING: Recall < 0.50. Consider raising top_k to 50 in the training loop below.")
else:
    print("Recall OK. Proceeding with top_k=30 for hard-negative mining.")

In [ ]:
# P2b — Step 2: Build training pairs and fine-tune CrossEncoder
# Positive pairs: (claim, gold_passage) → label 1
# Hard negatives: one BM25-retrieved non-gold passage per claim → label 0
from sentence_transformers import InputExample
from torch.utils.data import DataLoader as SentDataLoader

ce_train_samples = []
for item in train_data.values():
    claim    = item["claim_text"]
    gold_ids = set(item.get("evidences", []))
    if not gold_ids:
        continue
    for eid in gold_ids:
        if eid in evidence_dict:
            ce_train_samples.append(InputExample(texts=[claim, evidence_dict[eid]], label=1.0))
    candidates = retriever.retrieve(claim, top_k=20)
    for c in candidates:
        if c["id"] not in gold_ids:
            ce_train_samples.append(InputExample(texts=[claim, c["text"]], label=0.0))
            break  # one hard negative per claim

print(f"CrossEncoder training pairs: {len(ce_train_samples):,}")

import gc
gc.collect()

# Fine-tune CrossEncoder on CPU (22M params — no GPU needed)
ce_model = CrossEncoder("cross-encoder/ms-marco-MiniLM-L6-v2", num_labels=1)
ce_train_loader = SentDataLoader(ce_train_samples, shuffle=True, batch_size=16)
ce_model.fit(
    train_dataloader=ce_train_loader,
    epochs=3,
    warmup_steps=100,
    output_path="ce-finetuned/",
    show_progress_bar=True,
)
print("CrossEncoder fine-tuning complete. Saved to ce-finetuned/")

In [ ]:
# P2b — Step 3: Reload fine-tuned CrossEncoder into hybrid_retriever
# pipeline.predict() will now use the fine-tuned model for re-ranking.
# Invalidate oracle lookup cache so it's rebuilt from the updated retriever.
hybrid_retriever.cross_encoder = CrossEncoder("ce-finetuned/")
pipeline._ev_lookup = None  # force rebuild of oracle cache
print("hybrid_retriever updated with fine-tuned CrossEncoder.")
print("Re-run Section 3.1 to measure retrieval improvement.")

## 2.6 (Conditional) Priority 4 — LoRA Fine-Tuning

**Run this section only if gap analysis (Section 3.1) shows `A_oracle < 0.60` (classifier is the bottleneck).**

LoRA adds trainable low-rank adapter matrices to DeBERTa's attention layers, increasing
effective capacity without increasing VRAM. Performs on-par with full fine-tuning at a
fraction of the memory cost — `r=8` adds < 1% extra parameters (Hu et al., 2022).

After completing this section, the pipeline classifier is hot-swapped to the LoRA-adapted model.
Re-run Section 3 to measure the improvement.

In [ ]:
# P4 — LoRA fine-tuning: adds low-rank adapters to DeBERTa attention layers.
# Increases model capacity without raising VRAM beyond the T4 limit.
from peft import get_peft_model, LoraConfig, TaskType

# Reload a fresh base model for LoRA (avoids mixing with the full fine-tune above)
lora_base = AutoModelForSequenceClassification.from_pretrained(
    "MoritzLaurer/DeBERTa-v3-base-mnli-fever-anli",
    num_labels=4,
    id2label=ID2LABEL,
    label2id=LABEL2ID,
    ignore_mismatched_sizes=True,
).to(ft_device)

lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=8,
    lora_alpha=16,
    target_modules=["query_proj", "value_proj"],
    lora_dropout=0.1,
    bias="none",
)
lora_model = get_peft_model(lora_base, lora_config)
lora_model.print_trainable_parameters()

# Training loop — identical to Section 2.4 but using lora_model
lora_model = lora_model.float().to(ft_device)
lora_optimizer = AdamW(lora_model.parameters(), lr=2e-4, weight_decay=0.01)
lora_total_steps = len(train_loader) * EPOCHS
lora_scheduler = get_linear_schedule_with_warmup(
    lora_optimizer,
    num_warmup_steps  = lora_total_steps // 10,
    num_training_steps= lora_total_steps,
)
lora_scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
lora_loss_fn = torch.nn.CrossEntropyLoss(
    weight=train_dataset.class_weights.to(device=ft_device, dtype=torch.float32)
)

lora_best_f1 = 0.0
lora_no_improve = 0

for epoch in range(EPOCHS):
    lora_model.train()
    total_loss = 0.0
    for batch_enc, batch_labels in train_loader:
        batch_enc    = {k: v.to(ft_device) for k, v in batch_enc.items()}
        batch_labels = batch_labels.to(ft_device)
        lora_optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast(device_type=ft_device.type, dtype=torch.float16, enabled=use_amp):
            logits = lora_model(**batch_enc).logits
            loss   = lora_loss_fn(logits.float(), batch_labels)
        lora_scaler.scale(loss).backward()
        lora_scaler.unscale_(lora_optimizer)
        torch.nn.utils.clip_grad_norm_(lora_model.parameters(), 1.0)
        lora_scaler.step(lora_optimizer)
        lora_scaler.update()
        lora_scheduler.step()
        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)
    dev_f1   = _eval_macro_f1_dev(lora_model, ft_tokenizer, dev_data, evidence_dict, ft_device)
    print(f"Epoch {epoch+1}/{EPOCHS}  loss={avg_loss:.4f}  dev macro-F1={dev_f1:.4f}")

    if dev_f1 > lora_best_f1:
        lora_best_f1 = dev_f1
        lora_no_improve = 0
        lora_model.save_pretrained("lora-best")
        ft_tokenizer.save_pretrained("lora-best")
        print(f"  → New best ({lora_best_f1:.4f}). Checkpoint saved to lora-best/")
    else:
        lora_no_improve += 1
        if lora_no_improve >= patience:
            print(f"Early stop at epoch {epoch+1}. Best dev macro-F1: {lora_best_f1:.4f}")
            break

# Hot-swap LoRA model into pipeline classifier
from peft import PeftModel
lora_loaded = PeftModel.from_pretrained(lora_base, "lora-best").merge_and_unload()
classifier.model     = lora_loaded.half().to(classifier.device)
classifier.model.eval()
classifier.label_map = {
    "supports":        "SUPPORTS",
    "refutes":         "REFUTES",
    "not_enough_info": "NOT_ENOUGH_INFO",
    "disputed":        "DISPUTED",
}
pipeline._ev_lookup = None  # invalidate cache
print("Pipeline updated with LoRA-adapted classifier. Re-run Section 3.1 to measure improvement.")

# 3.Testing and Evaluation
(You can add as many code blocks and text blocks as you need. However, YOU SHOULD NOT MODIFY the section title)

## 3.1 Evaluate on Dev Set

In [ ]:
from sklearn.metrics import accuracy_score

LABEL_NAMES = ["SUPPORTS", "REFUTES", "NOT_ENOUGH_INFO", "DISPUTED"]

if dev_data:
    # ── Oracle mode (classifier ceiling — gold evidence, no retrieval) ─────────
    print(f"Running ORACLE eval on {len(dev_data):,} dev examples ...")
    oracle_preds, y_true_o, y_pred_o = {}, [], []
    for claim_id, item in dev_data.items():
        label, ev_ids = pipeline.predict(
            item["claim_text"],
            gold_evidence_ids=item.get("evidences")
        )
        oracle_preds[claim_id] = {"claim_label": label, "evidences": ev_ids}
        y_pred_o.append(label)
        y_true_o.append(item["claim_label"])
    print("Oracle eval done.\n")

    # ── Pipeline mode (real system — BM25 + CrossEncoder retrieval) ───────────
    print(f"Running PIPELINE eval on {len(dev_data):,} dev examples ...")
    pipeline_preds, y_true_p, y_pred_p = {}, [], []
    for i, (claim_id, item) in enumerate(dev_data.items()):
        label, ev_ids = pipeline.predict(item["claim_text"])
        pipeline_preds[claim_id] = {"claim_label": label, "evidences": ev_ids}
        y_pred_p.append(label)
        y_true_p.append(item["claim_label"])
        if (i + 1) % 50 == 0:
            print(f"  {i+1}/{len(dev_data)} done")
    print("Pipeline eval done.")
else:
    print("No dev data found.")
    oracle_preds = pipeline_preds = {}
    y_true_o = y_pred_o = y_true_p = y_pred_p = []

In [ ]:
if y_true_o and y_pred_o:
    print("=== ORACLE MODE (classifier ceiling — gold evidence) ===")
    print(classification_report(y_true_o, y_pred_o, labels=LABEL_NAMES, zero_division=0))

if y_true_p and y_pred_p:
    print("=== PIPELINE MODE (real system — BM25 + CrossEncoder retrieval) ===")
    print(classification_report(y_true_p, y_pred_p, labels=LABEL_NAMES, zero_division=0))

if y_true_o and y_true_p:
    a_oracle   = accuracy_score(y_true_o, y_pred_o)
    a_pipeline = accuracy_score(y_true_p, y_pred_p)
    gap        = a_oracle - a_pipeline

    print("=== GAP ANALYSIS ===")
    print(f"Oracle accuracy  : {a_oracle:.3f}  (classifier ceiling — gold evidence)")
    print(f"Pipeline accuracy: {a_pipeline:.3f}  (real system — retrieved evidence)")
    print(f"Retrieval gap    : {gap:.3f}  (oracle − pipeline)")
    print()
    if gap > 0.10:
        print("→ Gap > 0.10: RETRIEVER is the bottleneck.")
        print("  Run Section 2.5 (Priority 2b: supervised CrossEncoder fine-tuning),")
        print("  then re-run this evaluation to measure improvement.")
    elif a_oracle < 0.60:
        print("→ Gap ≤ 0.10 but oracle accuracy < 0.60: CLASSIFIER is the bottleneck.")
        print("  Run Section 2.6 (Priority 4: LoRA fine-tuning),")
        print("  then re-run this evaluation to measure improvement.")
    else:
        print("→ Gap ≤ 0.10 and oracle accuracy ≥ 0.60: system is well-balanced.")
        print("  Record scores in CLAUDE.md §3 and proceed to test predictions.")

In [ ]:
# Official eval.py score — harmonic mean of classification accuracy & evidence F-score
import importlib.util
from argparse import Namespace

if pipeline_preds:
    with open("dev-predictions.json", "w") as f:
        json.dump(pipeline_preds, f, indent=2)

    spec = importlib.util.spec_from_file_location("eval_module", "eval.py")
    eval_module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(eval_module)

    args = Namespace(predictions="dev-predictions.json",
                     groundtruth="dev-claims.json", verbose=False)
    eval_module.main(args)
else:
    print("No pipeline predictions available — run Section 3.1 first.")

## 3.2 Generate Test Predictions

In [ ]:
if test_data:
    print(f"Generating predictions for {len(test_data):,} test examples ...")
    test_predictions = {}

    for i, (claim_id, item) in enumerate(test_data.items()):
        label, ev_ids = pipeline.predict(item["claim_text"])
        test_predictions[claim_id] = {"claim_label": label, "evidences": ev_ids}
        if (i + 1) % 50 == 0:
            print(f"  {i+1}/{len(test_data)} done")

    with open("predictions.json", "w") as f:
        json.dump(test_predictions, f, indent=2)

    print(f"\nSaved {len(test_predictions)} predictions -> predictions.json")
    print(f"Label distribution: {dict(Counter(v['claim_label'] for v in test_predictions.values()))}")
else:
    print("No test data found.")

## Object Oriented Programming codes here

*All class definitions (`BM25Retriever`, `HybridRetriever`, `NLIClassifier`, `FactCheckPipeline`, `FactCheckDataset`) are in Section 0 above so the notebook runs correctly top-to-bottom with Run All.*